In [ ]:
import pandas as pd
import sqlite3
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

Chuẩn bị dữ liệu

In [ ]:
db_path = "../data/database/sccra_db.sqlite"
conn = sqlite3.connect(db_path)
df = pd.read_sql("SELECT [Consumer complaint narrative], Product FROM complaints WHERE [Consumer complaint narrative] IS NOT NULL LIMIT 20000", conn)
conn.close()

X = df['Consumer complaint narrative']
y = df['Product']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Feature Engineering (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

So sánh các thuật toán cho phân loại văn bản

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=-1)
}

results = []

for name, model in models.items():
    print(f"--- Đang huấn luyện {name} ---")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({"Model": name, "Accuracy": acc, "F1-Score": f1})

rdf = pd.DataFrame(results)
print(rdf)

Kết quả

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=rdf, x='Model', y='F1-Score', palette='Set2')
plt.title('So sánh F1-Score giữa các mô hình')
plt.ylim(0, 1.0)
plt.show()